# A4 · Prepare: cleaning, features, and the split by time

**Goal:** turn `raw_emr_export.csv` into model-ready data so that nothing from the future, and nothing recorded after booking, reaches the model.

```
raw ─► 1 duplicates ─► 2 labels ─► 3 age 999 ─► 4 minutes ─► 5 bad targets ─► 6 SPLIT BY TIME ─► 7 fit on train ─► 8 features
       └──────────────────── rules (row by row) ────────────────────┘                          └──── learned ────┘
```

**Before running this notebook,** write the `TODO(you)` functions and make the tests pass:

| File | Your functions | Test |
|---|---|---|
| `slotpred/clean.py` | `normalise_doctor`, `normalise_visit_type`, `fix_age`, `add_minutes`, `target_ok` | `.venv/Scripts/python.exe -m pytest tests/test_clean.py -v` |
| `slotpred/split.py` | `time_split` | `.venv/Scripts/python.exe -m pytest tests/test_split.py -v` |
| `slotpred/features.py` | `add_time_features`, `impute_age`, `fit_prep` | `.venv/Scripts/python.exe -m pytest tests/test_features.py -v` |

Then run the cells below. Sections 4 and 5 are your experiments.

> **A note on the test months.** Sections 4 and 5 look at July–August to *demonstrate* why the split matters. No setting is chosen from these numbers. From A5 onward the test split stays locked until A8.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from slotpred.clean import clean
from slotpred.eda import load_raw
from slotpred.features import FEATURES, fit_prep, make_features, make_xy
from slotpred.split import describe, random_split, time_split

raw = load_raw()
raw.shape

(43922, 16)

## 1 · Steps 1–5: the rules

`clean()` runs your rules in order and returns a **funnel**: how many rows each step removed.

In [2]:
df, funnel = clean(raw)
print(f"{len(raw):,} raw rows -> {len(df):,} clean visits")
funnel

43,922 raw rows -> 41,270 clean visits


,step,rows_in,removed,rows_out
0,1 drop exact duplicates,43922.0,175,43747.0
1,2 unrecognised doctor / visit type,43747.0,0,43747.0
2,5a consult_end missing,NaN,100,NaN
3,5b Whitefield Mar-Apr process change,NaN,1740,NaN
4,5c under 1 or over 120 min,NaN,637,NaN
5,5 total untrustworthy targets,43747.0,2477,41270.0


In [3]:
from slotpred.eda import load_raw
from slotpred.clean import normalise_labels, add_minutes, GLITCH_FROM, GLITCH_TO

raw = load_raw()

def in_window(d):
    return (d.clinic == "Whitefield") & d.date.between(GLITCH_FROM, GLITCH_TO)

# ---- Row 7: Whitefield Mar-Apr ----
print("A3 view (raw):", in_window(raw).sum())                      # 1753
print("...of which duplicate rows:", in_window(raw[raw.duplicated()]).sum())   # 9

df_before = normalise_labels(raw.drop_duplicates().reset_index(drop=True))
df_before = df_before.assign(minutes=add_minutes(df_before))       # table before step 5

w = in_window(df_before)
print("after dedup:", w.sum())                                     # 1744
print("...missing minutes:", (w & df_before.minutes.isna()).sum()) # 4 -> counted under 5a
print("A4 removed as 5b:", (w & df_before.minutes.notna()).sum())  # 1740

# ---- Row 6: under 1 or over 120 ----
m_raw = add_minutes(raw)
oor_raw = m_raw.notna() & ~m_raw.between(1, 120)
print("A3 view (raw):", oor_raw.sum())                             # 669
print("...of which duplicates:", (oor_raw & raw.duplicated()).sum())  # 3

oor = df_before.minutes.notna() & ~df_before.minutes.between(1, 120)
print("after dedup:", oor.sum())                                   # 666
print("...also in Whitefield window (counted under 5b):", (oor & w).sum())  # 29
print("A4 removed as 5c:", (oor & ~w).sum())                       # 637

A3 view (raw): 1753
...of which duplicate rows: 9
after dedup: 1744
...missing minutes: 4
A4 removed as 5b: 1740
A3 view (raw): 669
...of which duplicates: 3
after dedup: 666
...also in Whitefield window (counted under 5b): 29
A4 removed as 5c: 637


**TODO(you): reconcile the funnel with your A3 problem log.** The counts won't all match exactly. For each row, find out why.

| Log # | Problem | A3 count | A4 removed | Same? If not, why? |
|---|---|---|---|---|
| 1 | Duplicates | 175 |175 | same|
| 5 | Missing `consult_end` | 100 | 100|same |
| 7 | Whitefield March–April | 1,753 |1740 |no,9 were duplicates (removed in step 1) and 4 had missing consult_end (counted under 5a) |
| 6 | Under 1 or over 120 min | 669 |637 | no, 3 were duplicates and 29 were also in the Whitefield window, so they're counted under 5b. A4 removed 637|

*Hint:* the funnel counts a row with two problems only once, under the first reason. Try `(df_before.clinic == "Whitefield") & ...` on the table before step 5. You can rebuild it with `from slotpred.clean import normalise_labels, add_minutes`.

In [ ]:
# Row 7: "no. 9 were duplicates (removed in step 1) and 4 had missing consult_end (counted under 5a)."
# Row 6: "no. 3 were duplicates and 29 were also in the Whitefield window, so they're counted under 5b. A4 removed 637."

## 2 · Step 6: split by time

In [4]:
splits = time_split(df)
describe(splits)

,rows,first,last
train,24818,2026-01-01,2026-05-30
valid,5515,2026-06-01,2026-06-30
test,10937,2026-07-01,2026-08-31


**Question:** the training split ends on 30 May, not 31 May. Why? *(Hint: `pd.Timestamp("2026-05-31").day_name()`.)*

*Answer:* because its a sunday

In [5]:

#TODO(you): in which split(s) does Dr Nikhil Verma appear, and how many visits in each?
# Dr Nikhil Verma doent appear anywhere as we alreadybhave normalised the doctor names hence Nikhil Verma is used instead
# it appears in test split for 500 times and rest 0
pd.Series({name: (part["doctor"] == "Dr Nikhil Verma").sum() for name, part in splits.items()})
df["doctor"].unique()
pd.Series({name: (part["doctor"] == "Nikhil Verma").sum() for name, part in splits.items()})

train      0
valid      0
test     500
dtype: int64

## 3 · Steps 7–8: fit on train, then build features for every split

`fit_prep` sees **only** `splits["train"]`. `make_features` applies what it learned, unchanged, to all three splits.

In [6]:
state = fit_prep(splits["train"])
print("age_fill =", state.age_fill)
print({k: len(v) for k, v in state.categories.items()})

X_train, y_train = make_xy(splits["train"], state)
X_valid, y_valid = make_xy(splits["valid"], state)
X_test, y_test = make_xy(splits["test"], state)
display(X_train.head())
X_train.dtypes

age_fill = 42.0
{'department': 8, 'visit_type': 3, 'doctor': 20, 'clinic': 5}


C:\Users\siddh\Documents\slot-prediction-model\slotpred\features.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["hour"] = pd.to_datetime(out["slot_time"], errors="coerce").dt.hour
C:\Users\siddh\Documents\slot-prediction-model\slotpred\features.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["hour"] = pd.to_datetime(out["slot_time"], errors="coerce").dt.hour
C:\Users\siddh\Documents\slot-prediction-model\slotpred\features.py:41: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  out["hour"] = pd.to_datetime(out["slot_time"], errors="coerce").dt.hour


,department,visit_type,doctor,clinic,age,age_missing,interpreter,teleconsult,prior_reports,weekday,hour
1,Cardiology,follow-up,Rahul Menon,Koramangala,30.0,0,0,0,0,0,11
3,General Medicine,new,Imran Khan,Indiranagar,43.0,0,0,0,1,0,17
6,ENT,review,Rohan Desai,HSR,51.0,0,0,1,4,0,15
7,ENT,new,Vikram Shetty,Indiranagar,42.0,0,0,0,0,2,16
8,Orthopaedics,new,Arjun Reddy,HSR,52.0,0,1,0,0,5,17


department       category
visit_type       category
doctor           category
clinic           category
age               float64
age_missing         int64
interpreter         int64
teleconsult         int64
prior_reports       int64
weekday             int32
hour                int32
dtype: object

In [ ]:
# TODO(you): check two things in X_test
# 1. What happened to Nikhil Verma's rows in the `doctor` column? (compare splits["test"]["doctor"] with X_test["doctor"])
nikhil_test = splits["test"]["doctor"] == "Nikhil Verma"
print("Nikhil Verma rows in test:", nikhil_test.sum())
display(pd.DataFrame({
    "source_doctor": splits["test"].loc[nikhil_test, "doctor"],
    "feature_doctor": X_test.loc[nikhil_test, "doctor"],
}))
# in feature_splits Nikhil verma is not present in the `doctor` column of X_test anymore

# 2. How many rows have age_missing == 1 in each split, and what is their `age` now?
feature_splits = {"train": X_train, "valid": X_valid, "test": X_test}
for split_name, features in feature_splits.items():
    missing_age = features["age_missing"] == 1
    print(f"{split_name} - age_missing == 1: {missing_age.sum()}")
    display(features.loc[missing_age, ["age"]])

# in feature_splits, rows with age_missing == 1 have their `age` displayed above,for train its 494, for valid its 136, and for test its 221.and age is 42 

Nikhil Verma rows in test: 500


,source_doctor,feature_doctor
5,Nikhil Verma,NaN
162,Nikhil Verma,NaN
422,Nikhil Verma,NaN
711,Nikhil Verma,NaN
834,Nikhil Verma,NaN
...,...,...
41116,Nikhil Verma,NaN
41137,Nikhil Verma,NaN
41182,Nikhil Verma,NaN
41205,Nikhil Verma,NaN


train - age_missing == 1: 494


,age
26,42.0
71,42.0
131,42.0
132,42.0
178,42.0
...,...
40994,42.0
41054,42.0
41078,42.0
41195,42.0


valid - age_missing == 1: 136


,age
505,42.0
570,42.0
874,42.0
1431,42.0
1877,42.0
...,...
40785,42.0
40792,42.0
40943,42.0
41160,42.0


test - age_missing == 1: 221


,age
45,42.0
240,42.0
288,42.0
293,42.0
573,42.0
...,...
40017,42.0
40245,42.0
40546,42.0
40827,42.0


## 4 · Experiment 1: random split vs split by time

The simplest sensible model: **predict the median `minutes` of the visit's department × visit type**, learned from the training rows. (This is A5's baseline; here it's just a measuring stick.)

The helper is given. You run it on both kinds of split.

In [9]:
def median_baseline(train, test, key=("department", "visit_type")):
    """Fit group medians on `train`, score on `test`. Returns overall MAE, Dermatology MAE and Dermatology bias."""
    key = list(key)
    medians = train.groupby(key)["minutes"].median().rename("pred")
    t = test.join(medians, on=key)
    t["pred"] = t["pred"].fillna(train["minutes"].median())       # groups never seen in train
    err = t["minutes"] - t["pred"]
    derm = t["department"] == "Dermatology"
    return pd.Series({"MAE": err.abs().mean(), "Derm MAE": err[derm].abs().mean(),
                      "Derm bias (actual - pred)": err[derm].mean(), "test rows": len(t)}).round(2)

In [10]:
# TODO(you): build this table
#   row "time split":   fit on splits["train"], score on splits["test"]
#   row "random split": r = random_split(df); fit on r["train"], score on r["test"]
#   row "random-split model, scored on Jul-Aug only": fit on r["train"], score on the rows of r["test"] dated >= "2026-07-01"
# hint: pd.DataFrame({"time split": median_baseline(...), ...}).T

time_scores = median_baseline(splits["train"], splits["test"])
random_split_result = random_split(df)
random_split_score = median_baseline(random_split_result["train"], random_split_result["test"])
random_split_jul_aug = median_baseline(
    random_split_result["train"],
    random_split_result["test"][random_split_result["test"]["date"] >= "2026-07-01"]
)
pd.DataFrame({
    "time split": time_scores,
    "random split": random_split_score,
    "random-split model, scored on Jul-Aug only": random_split_jul_aug,
}).T

,MAE,Derm MAE,Derm bias (actual - pred),test rows
time split,3.83,4.13,3.66,10937.0
random split,3.54,3.10,0.83,8254.0
"random-split model, scored on Jul-Aug only",3.84,4.19,3.29,2234.0


**Questions**
1. Which split makes the model look better overall? Which one tells you what will happen after deployment?
2. Dermatology bias: why is it about +3.7 with the split by time and much smaller with the random split? Connect it to the simulator (A2).
3. Rows 2 and 3 are the *same* model. Why is its Dermatology bias so different on the two test sets? So where did the random split's good score come from?
4. Row 3's bias is a little smaller than row 1's. What leaked into the random-split model?

*(Your numbers differ slightly from the lesson's, because the random split depends on its seed.)*

*Answers:*
1. The random split looks better, with MAE 3.54 against 3.83.
The time split is the one that tells you what happens after deployment, because you train on the past and predict a future the model hasn't seen. 
2. Bias here is actual minus predicted, so +3.7 means the model under-predicts Dermatology by about 3.7 minutes.
In the simulator (A2), Dermatology gets drift from 1 July. It's an additive +3 minutes (DRIFT_EXTRA_MIN), applied after the doctor's pace multiplier.
The time split trains on Jan–May, before the drift, so the median never sees the extra 3 minutes. It then predicts Jul–Aug, where every Dermatology visit takes about 3 minutes longer. That gap is the +3.7 bias, roughly the +3 drift plus some noise.
This is concept drift: the same inputs now give a longer outcome.
With the random split the bias is only +0.83, because the training data already includes post-drift rows.
3. Rows 2 and 3 are one model, trained on the random-split train set, which is a mix of pre-drift and post-drift months.
Row 2 is scored on a random test set that is mostly pre-drift (Jan–Jun). There the drift doesn't apply, so the model looks accurate and unbiased.
Row 3 is scored only on Jul–Aug, where the drift is active. The bias jumps to +3.29 and the Derm MAE to 4.19.
So the random split's good score came from a test set that mostly contained easy, pre-drift months. It hid the drift rather than handling it. Scored on Jul–Aug only, MAE is 3.84, the same as the honest 3.83.
4. The random split put some Jul–Aug rows into training, so the model saw a few post-drift Dermatology visits. That is future information leaking into training.
The Dermatology median therefore moved up slightly, which cuts the under-prediction from 3.66 to 3.29.
The leak is small, because most training rows are still pre-drift. That is why the bias only shrinks a little instead of vanishing.

## 5 · Experiment 2: what is the learned step allowed to see?

Same helper, a finer key: the median **per doctor × visit type**. Fit it two ways and score both on the test months.

In [11]:
# TODO(you):
# 1. honest: fit on splits["train"]                       -> score on splits["test"]
# 2. leaky:  fit on df (all months, including the test months) -> score on splits["test"]
# 3. The same idea for age: compare splits["train"]["age"].median() with df["age"].median()

honest = median_baseline(splits["train"], splits["test"])
leaky = median_baseline(df, splits["test"])
train_age_median = splits["train"]["age"].median()
overall_age_median = df["age"].median()

honest, leaky, train_age_median, overall_age_median

(MAE                              3.83
 Derm MAE                         4.13
 Derm bias (actual - pred)        3.66
 test rows                    10937.00
 dtype: float64,
 MAE                              3.77
 Derm MAE                         3.72
 Derm bias (actual - pred)        2.99
 test rows                    10937.00
 dtype: float64,
 np.float64(42.0),
 np.float64(42.0))

**Questions**
1. How much better does the leaky version *look*? Would that improvement exist after deployment?
2. The age median barely moves (or doesn't move at all). Does that make "fit on train only" unnecessary for age? Why or why not?
3. Which kind of statistic leaks the most: ones computed from the inputs (age) or from the target (`minutes`)? Why?

*Answers:*
1. Overall MAE improves only slightly, from 3.83 to 3.77 (about 0.06 min).
Dermatology improves more: MAE falls from 4.13 to 3.72, and bias falls from 3.66 to 2.99.
The gain is mostly in Dermatology because the leaky fit has seen the Jul–Aug drift (+3 min) in the test months.
The improvement would not exist after deployment. In production, future visits aren't available when you compute the median, so the model would behave like the honest row. The leaky number is an illusion.
2. No. The rule must be followed regardless of whether the number happens to change.
Here age didn't drift, so the leak was harmless this time. That is luck about this dataset, not a property of the method.
If the patient mix changed (for example, a new clinic with older patients), the all-data median would differ from the train median, and the leak would be real.A pipeline has to be safe by construction if we skip the rule when it looks fine then eventually will skip when it maatters
3. Target-derived statistics leak the most.
The target is exactly what you're trying to predict. A median of minutes over all months directly encodes the future answers, including drift. That's why Dermatology moved so much in the leaky row.
An input statistic like the age median only describes the features. It leaks future information about who shows up, but it doesn't contain the outcome, so its effect on error is usually indirect and small.
Both are still leaks and both should be fit on train only. Target leakage is just the more damaging one, because it feeds the answer into the model.

## 6 · Save the prepared data

One command reruns the whole pipeline and saves the splits and `prep_state.json` for A5 onward. You can also run it from the terminal: `.venv/Scripts/python.exe -m slotpred.prepare`.

In [12]:
from slotpred import prepare
splits, state = prepare.main()

                                step  rows_in  removed  rows_out
             1 drop exact duplicates  43922.0      175   43747.0
  2 unrecognised doctor / visit type  43747.0        0   43747.0
              5a consult_end missing      NaN      100       NaN
5b Whitefield Mar-Apr process change      NaN     1740       NaN
          5c under 1 or over 120 min      NaN      637       NaN
       5 total untrustworthy targets  43747.0     2477   41270.0 

        rows       first        last
train  24818  2026-01-01  2026-05-30
valid   5515  2026-06-01  2026-06-30
test   10937  2026-07-01  2026-08-31 

age_fill = 42.0; levels: {'department': 8, 'visit_type': 3, 'doctor': 20, 'clinic': 5}
wrote C:\Users\siddh\Documents\slot-prediction-model\data\prepared


## Wrap-up questions (answer in `notes/A04-prepare.md` too)

1. Give one example of a **rule** and one of a **learned step** from this lesson. Where in the pipeline may each run?
2. Why is `target_ok` never called by the API, while `normalise_doctor` must be?
3. Why do we keep `doctor` as a feature even though Nikhil Verma is unseen?
4. What would go wrong if you dropped duplicates *after* a random split?